# 07 The 1.85x deprivation ratio
The release says A&E attendance rates for the most deprived areas were 1.85 times those of the least deprived (source: ECDS). I rebuild it from the ECDS National Report Tables. The 13.6 MB workbook is not in the repository; if it is absent this notebook reads the small Silver table it wrote the first time, and says so.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import common as c
pd.options.display.float_format = '{:,.4f}'.format
pd.options.display.width = 200


In [2]:
if c.ECDS_NATIONAL.exists():
    imd = c.read_ecds_imd(); imd.to_parquet(c.SILVER / 'ecds_imd_decile.parquet', index=False); print('read from Bronze workbook')
else:
    imd = pd.read_parquet(c.SILVER / 'ecds_imd_decile.parquet'); print('Bronze workbook not present; read the committed Silver table')
imd[imd.period == '2025/26']

read from Bronze workbook


,period,group,attendances,population,rate_per_100k
0,2025/26,Least deprived 10%,1846593,"5,484,839.0000","33,667.0000"
1,2025/26,Less deprived 10-20%,2050001,"5,632,520.0000","36,396.0000"
2,2025/26,Less deprived 20-30%,2180954,"5,656,824.0000","38,554.0000"
3,2025/26,Less deprived 30-40%,2276301,"5,740,722.0000","39,652.0000"
4,2025/26,Less deprived 40-50%,2428547,"5,696,114.0000","42,635.0000"
5,2025/26,More deprived 40-50%,2564002,"5,779,606.0000","44,363.0000"
6,2025/26,More deprived 30-40%,2775384,"5,782,194.0000","47,999.0000"
7,2025/26,More deprived 20-30%,3005200,"5,839,393.0000","51,464.0000"
8,2025/26,More deprived 10-20%,3244141,"5,764,540.0000","56,278.0000"
9,2025/26,Most deprived 10%,3572235,"5,735,377.0000","62,284.0000"


In [3]:
# Internal check: the ten deciles plus the unknown-IMD attendances must sum to the published total.
cur = imd[imd.period == '2025/26'].set_index('group')
deciles = cur.drop(['Unknown', 'IMD_DECILE_TOTAL'])
print('deciles + unknown =', int(deciles.attendances.sum() + cur.loc['Unknown', 'attendances']), '| published total', int(cur.loc['IMD_DECILE_TOTAL', 'attendances']))
assert deciles.attendances.sum() + cur.loc['Unknown', 'attendances'] == cur.loc['IMD_DECILE_TOTAL', 'attendances']
print('unknown-IMD share of attendances: %.2f%%' % (100 * cur.loc['Unknown', 'attendances'] / cur.loc['IMD_DECILE_TOTAL', 'attendances']))
print('rate rises at every step from least to most deprived:', deciles.rate_per_100k.is_monotonic_increasing)

deciles + unknown = 26639584 | published total 26639584
unknown-IMD share of attendances: 2.61%
rate rises at every step from least to most deprived: True


In [4]:
rows = []
for yr, g in imd.groupby('period'):
    g = g.set_index('group'); mo, le = g.loc['Most deprived 10%'], g.loc['Least deprived 10%']
    rows.append({'year': yr, 'most_attendances': mo.attendances, 'least_attendances': le.attendances,
                 'most_population': mo.population, 'least_population': le.population,
                 'ratio_published_rates': mo.rate_per_100k / le.rate_per_100k,
                 'ratio_rebuilt_from_counts': (mo.attendances / mo.population) / (le.attendances / le.population),
                 'ratio_of_raw_counts': mo.attendances / le.attendances})
ratio = pd.DataFrame(rows)
ratio.round(4)

,year,most_attendances,least_attendances,most_population,least_population,ratio_published_rates,ratio_rebuilt_from_counts,ratio_of_raw_counts
0,2020/21,2188353,1125242,"5,615,307.0000","5,423,146.0000",1.8782,1.8782,1.9448
1,2021/22,3018316,1546722,"5,615,307.0000","5,423,146.0000",1.8846,1.8846,1.9514
2,2022/23,3106073,1564465,"5,615,307.0000","5,423,146.0000",1.9174,1.9174,1.9854
3,2023/24,3271694,1668418,"5,615,307.0000","5,423,146.0000",1.8938,1.8939,1.9610
4,2024/25,3468649,1769132,"5,615,307.0000","5,423,146.0000",1.8935,1.8936,1.9607
5,2025/26,3572235,1846593,"5,735,377.0000","5,484,839.0000",1.8500,1.8500,1.9345


In [5]:
# The gate for this notebook: the published statement is 1.85, and I expect to reproduce it from the counts as well as the rates.
r = ratio.set_index('year').loc['2025/26']
print('published-rate ratio %.4f | rebuilt from counts %.4f' % (r.ratio_published_rates, r.ratio_rebuilt_from_counts))
assert round(r.ratio_published_rates, 2) == 1.85 and abs(r.ratio_published_rates - r.ratio_rebuilt_from_counts) < 1e-3

published-rate ratio 1.8500 | rebuilt from counts 1.8500


## The 2025/26 population update
The decile populations changed in 2025/26, so part of the apparent fall from 2024/25 is a denominator effect. I hold the populations constant in each direction to see how much.

In [6]:
a, b = ratio.set_index('year').loc['2024/25'], ratio.set_index('year').loc['2025/26']
def rr(att_m, pop_m, att_l, pop_l): return (att_m / pop_m) / (att_l / pop_l)
sens = pd.Series({
    '2024/25 counts, 2024/25 populations': rr(a.most_attendances, a.most_population, a.least_attendances, a.least_population),
    '2024/25 counts, 2025/26 populations': rr(a.most_attendances, b.most_population, a.least_attendances, b.least_population),
    '2025/26 counts, 2024/25 populations': rr(b.most_attendances, a.most_population, b.least_attendances, a.least_population),
    '2025/26 counts, 2025/26 populations': rr(b.most_attendances, b.most_population, b.least_attendances, b.least_population)})
sens.round(4)

2024/25 counts, 2024/25 populations   1.8936
2024/25 counts, 2025/26 populations   1.8750
2025/26 counts, 2024/25 populations   1.8683
2025/26 counts, 2025/26 populations   1.8500
dtype: float64

In [7]:
# The outpatient file uses the same IMD populations; I confirm that rather than assume it.
op5 = pd.read_excel(c.OP_REPORT, sheet_name='Summary Report 5', header=7).iloc[:10]
op_pop = op5.set_index(op5.columns[0])['National population per IMD decile']
print('most deprived  ECDS', int(b.most_population), '| outpatient', int(op_pop['Most deprived 10%']))
print('least deprived ECDS', int(b.least_population), '| outpatient', int(op_pop['Less deprived 10%']))
assert b.most_population == op_pop['Most deprived 10%'] and b.least_population == op_pop['Less deprived 10%']

most deprived  ECDS 5735377 | outpatient 5735377
least deprived ECDS 5484839 | outpatient 5484839


In [8]:
ratio.to_parquet(c.GOLD / 'deprivation_ratio_by_year.parquet', index=False)
sens.rename('ratio').reset_index().rename(columns={'index': 'scenario'}).to_parquet(c.GOLD / 'deprivation_ratio_population_sensitivity.parquet', index=False)